In [0]:
import datetime
# 本脚本用于批量回滚 Delta 表到指定历史版本。
# 新逻辑：
#   1. 对 FULL_TABLE_NAMES 中的每张表执行 DESC HISTORY，展示完整历史。
#   2. 按 timestamp 字段找到 2026-06-23 当天 0 点之前（即 2026-06-22 及更早）
#      的最后一个版本，作为目标回滚版本。
#   3. 展示目标版本对应的历史记录，并打印 RESTORE SQL。
#   4. 如果 AUTO_RESTORE=True，则执行 RESTORE TABLE ... TO VERSION AS OF <目标版本>。
# 设计约束：
#   - 在 cutoff 日期之前没有任何版本的表会被跳过并告警。
#   - STOP_ON_ERROR=False 时，单表失败不会中断后续表的处理。

# 全局开关：True 自动执行 RESTORE；False 仅打印目标版本和 SQL（预览模式）
AUTO_RESTORE = False

# 全局开关：True 时任意表失败会中断整个 notebook；False 会跳过失败表继续处理
STOP_ON_ERROR = True

# 回滚截止日期（不含当天）：目标版本必须严格早于该日期
CUTOFF_DATE = datetime.date(2026, 6, 23)

# 从 catalog.csv 解析出的全表名列表（catalog.schema.table）。
# 如果后续需要新增/删除表，直接修改这个列表即可。
FULL_TABLE_NAMES = [
    "catalog_southeastasia_mdm_pr.consumer_master.t_master_consumer",
    "catalog_southeastasia_mdm_pr.consumer_master.t_master_address",
    "catalog_southeastasia_mdm_pr.consumer_master.t_master_auxiliary_attribute",
    "catalog_southeastasia_mdm_pr.consumer_master.t_master_consumer_group",
    "catalog_southeastasia_mdm_pr.consumer_master.t_master_crossbrand_optin",
    "catalog_southeastasia_mdm_pr.consumer_master.t_master_custom_attributes",
    "catalog_southeastasia_mdm_pr.consumer_master.t_master_emedia",
    "catalog_southeastasia_mdm_pr.consumer_master.t_master_hair_concerns",
    "catalog_southeastasia_mdm_pr.consumer_master.t_master_hair_type",
    "catalog_southeastasia_mdm_pr.consumer_master.t_master_hobby",
    "catalog_southeastasia_mdm_pr.consumer_master.t_master_makeup_concerns",
    "catalog_southeastasia_mdm_pr.consumer_master.t_master_notes",
    "catalog_southeastasia_mdm_pr.consumer_master.t_master_optin",
    "catalog_southeastasia_mdm_pr.consumer_master.t_master_phone",
    "catalog_southeastasia_mdm_pr.consumer_master.t_master_program",
    "catalog_southeastasia_mdm_pr.consumer_master.t_master_remark",
    "catalog_southeastasia_mdm_pr.consumer_master.t_master_skin_concerns",
    "catalog_southeastasia_mdm_pr.consumer_master.t_master_terms",
    "catalog_southeastasia_mdm_pr.consumer_master.t_transaction_master",
    "catalog_southeastasia_mdm_pr.consumer_master.t_sconsumermedia_line_unbind_records",
    "catalog_southeastasia_mdm_pr.consumer_combine.t_cbr_dataset",
    "catalog_southeastasia_mdm_pr.consumer_combine.t_cbr_withoutPII_dataset",
    "catalog_southeastasia_mdm_pr.consumer_combine.t_cbrdrjart_dataset",
]


# 以下逻辑原位于 .context/parallel_run/truncate_tables.py，现合并到本脚本末尾，
# 用于在 restore 之后批量 truncate 指定的 Delta 表。
# 表名从 env.py 中的 schema 推断：
#   - consumer_combine: c_cbr_dataset, c_cbr_withoutPII_dataset, c_cbrdrjart_dataset,
#                       c_transaction_master_dataset, c_membership_mapping_log,
#                       t_transaction_master_dataset, t_membership_mapping_log
#   - consumer_master: t_derived_consumer_l1, t_derived_consumer_l2, t_derived_consumer_l3
# 统一 catalog: catalog_southeastasia_mdm_pr

# 全局开关：True 实际执行 TRUNCATE；False 只打印将要执行的 SQL（预览模式）
EXECUTE_TRUNCATE = False

# 全局开关：True 时任意表失败会中断整个 notebook；False 会跳过失败表继续处理
STOP_ON_ERROR = True

# 需要 truncate 的全表名列表（catalog.schema.table）
TABLES_TO_TRUNCATE = [
    # consumer_combine 下的表（对应 env.py 中的 golden_consumer_combine_database）
    "catalog_southeastasia_mdm_pr.consumer_combine.c_cbr_dataset",
    "catalog_southeastasia_mdm_pr.consumer_combine.c_cbr_withoutPII_dataset",
    "catalog_southeastasia_mdm_pr.consumer_combine.c_cbrdrjart_dataset",
    "catalog_southeastasia_mdm_pr.consumer_combine.c_transaction_master_dataset",
    "catalog_southeastasia_mdm_pr.consumer_combine.c_membership_mapping_log",
    "catalog_southeastasia_mdm_pr.consumer_combine.t_transaction_master_dataset",
    "catalog_southeastasia_mdm_pr.consumer_combine.t_membership_mapping_log",
    # consumer_master 下的表（对应 env.py 中的 golden_consumer_master_database）
    "catalog_southeastasia_mdm_pr.consumer_master.t_derived_consumer_l1",
    "catalog_southeastasia_mdm_pr.consumer_master.t_derived_consumer_l2",
    "catalog_southeastasia_mdm_pr.consumer_master.t_derived_consumer_l3",
]

In [0]:
def _find_target_version(history_rows, cutoff_date):
    """
    从历史版本列表中找到 cutoff_date 当天 0 点之前的最新版本。

    规则：
      - 遍历所有历史版本，比较 timestamp 与 cutoff_date。
      - timestamp < cutoff_date 0 点的版本中，version 最大的即为目标版本。
      - 如果没有符合条件的版本，返回 None。

    参数：
      history_rows: list[dict|Row]，每个元素包含 version 和 timestamp 字段。
      cutoff_date: datetime.date，截止日期（不含当天）。

    返回：
      int | None: 目标版本号，找不到时返回 None。
    """
    if not history_rows:
        return None

    # 将截止日期统一为 UTC 的 datetime
    cutoff = datetime.datetime(
        cutoff_date.year, cutoff_date.month, cutoff_date.day,
        tzinfo=datetime.timezone.utc
    )

    target_version = None
    for row in history_rows:
        ts = row["timestamp"]
        # 统一把字符串 / date / datetime 转成带时区的 datetime
        if isinstance(ts, str):
            ts = datetime.datetime.fromisoformat(ts.replace("Z", "+00:00"))
        elif isinstance(ts, datetime.date) and not isinstance(ts, datetime.datetime):
            ts = datetime.datetime.combine(ts, datetime.time.min)
        if ts.tzinfo is None:
            ts = ts.replace(tzinfo=datetime.timezone.utc)

        if ts < cutoff:
            v = int(row["version"])
            if target_version is None or v > target_version:
                target_version = v

    return target_version

def _run_self_check():
    """
    内存中的单元测试，验证 _find_target_version 的核心逻辑。
    通过 widget run_self_check 控制是否执行，默认不执行。
    """
    rows = [
        {"version": 0, "operation": "CREATE TABLE AS SELECT", "timestamp": datetime.datetime(2026, 6, 21)},
        {"version": 1, "operation": "WRITE", "timestamp": datetime.datetime(2026, 6, 22, 23, 59)},
        {"version": 2, "operation": "MERGE", "timestamp": datetime.datetime(2026, 6, 23, 1, 0)},
        {"version": 3, "operation": "MERGE", "timestamp": datetime.datetime(2026, 6, 24)},
    ]
    # 2026-06-23 之前的最后一个版本是 1
    assert _find_target_version(rows, datetime.date(2026, 6, 23)) == 1
    # 没有历史时返回 None
    assert _find_target_version([], datetime.date(2026, 6, 23)) is None
    # 全部版本都在 cutoff 之后（含当天 0 点）时返回 None
    assert _find_target_version(
        [{"version": 0, "operation": "WRITE", "timestamp": datetime.datetime(2026, 6, 23, 0, 0)}],
        datetime.date(2026, 6, 23)
    ) is None
    print("self-check passed")

def _process_table(full_name):
    """
    处理单张表：查询历史、定位目标版本、展示结果、打印 SQL、执行回滚。

    参数：
      full_name: str, 完整表名，格式 catalog.schema.table
    """
    from pyspark.sql import functions as F

    print(f"Processing {full_name}")
    try:
        # 1. 查询 Delta 历史版本并展示
        history_df = spark.sql(f"DESC HISTORY {full_name}")
        display(history_df.orderBy("version"))

        # 2. 把历史记录收集到 driver，计算目标回滚版本
        rows = [r.asDict(recursive=True) for r in history_df.collect()]
        target_version = _find_target_version(rows, CUTOFF_DATE)

        # 3. 找不到目标版本时跳过
        if target_version is None:
            print(f"WARN: No version before {CUTOFF_DATE} found for {full_name}; skipping.")
            return

        # 4. 展示目标版本对应的历史记录
        target_row = history_df.filter(F.col("version") == target_version)
        display(target_row)

        # 5. 打印并执行 RESTORE SQL
        restore_sql = f"RESTORE TABLE {full_name} TO VERSION AS OF {target_version}"
        print(f"Restore SQL: {restore_sql}")

        if AUTO_RESTORE:
            spark.sql(restore_sql)
            print(f"RESTORED {full_name} to version {target_version}")
        else:
            print(f"Skipped execution (AUTO_RESTORE=False)")

    except Exception as e:
        # 单表异常处理：打印错误，并根据 STOP_ON_ERROR 决定是否抛出
        print(f"ERROR processing {full_name}: {e}")
        if STOP_ON_ERROR:
            raise

def _truncate_table(full_name):
    """
    对单张表执行 TRUNCATE。

    参数：
      full_name: str, 完整表名，格式 catalog.schema.table
    """
    print(f"Processing {full_name}")
    try:
        if EXECUTE_TRUNCATE:
            # 执行 TRUNCATE TABLE，清空表中所有数据，保留表结构
            spark.sql(f"TRUNCATE TABLE {full_name} ")
            print(f"TRUNCATED {full_name}")
        else:
            print(f"Would truncate {full_name} (EXECUTE_TRUNCATE=False)")
    except Exception as e:
        # 单表异常处理：打印错误，并根据 STOP_ON_ERROR 决定是否抛出
        print(f"ERROR processing {full_name}: {e}")
        if STOP_ON_ERROR:
            raise

In [0]:
# 主循环：逐个处理 FULL_TABLE_NAMES 中的表
for full_name in FULL_TABLE_NAMES:
    _process_table(full_name)

In [0]:
# 主循环：逐个 truncate TABLES_TO_TRUNCATE 中的表
for full_name in TABLES_TO_TRUNCATE:
    _truncate_table(full_name)

In [0]:
# %sql

# update 
#     catalog_southeastasia_mdm_pr.share_mdm_config.t_topic_batch_log
# set
#     is_handle = false

In [0]:
%sql

select
    topic, is_handle, count(*), sum(read_record_count)
from
    catalog_southeastasia_mdm_pr.share_mdm_config.t_topic_batch_log
group by
    topic, is_handle
    


